# 47 · Revision addenda: integration-gene circularity, transporter shares, Table S2 with coordinates

Three checks requested after draft v0.3 (referee M1 fix 3 and M8):

| Part | Question |
|---|---|
| (a) | The reviewed labels come from clustering on notebook 13's 611 integration (PCA) genes, which are over-represented among conserved genes. Does the conservation index stay low without them? |
| (b) | What share of zonated transporter genes (Slc/Abc/Aqp) is conserved, against all genes? |
| (c) | Table S2 with each pathway's robustness under the DPT13 and SCF13-ED coordinates |
| (b′) | Transporters as direction agreement among genes zonated in both species |
| (d) | Integration-gene enrichment under notebook 43's symmetric classes |
| (e) | Which joint test calls each robust pathway, and its peak in segment units under three coordinates |
| (f) | Reversal counts under reviewed, transferred, cross-fit and no-integration-gene settings |

**Protocol** (workstream-1 plan, addenda round, fixed before any result). (a) reuses notebook 46's
evaluation function unchanged, with the reviewed labels and notebook 46's saved cross-fit label
files for settings (i) and (ii). Evaluation genes exclude the 611 integration genes. The claim
"largely species-specific" stands if the primary index is ≤ 0.5 with a 95% upper bound < 0.6 for
both S2 − S1 and S3 − early in every evaluation. Two mice and two cortex sections from one human
donor; descriptive.

## 1 · Folders and settings (notebook 43's constants)

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
reference43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_revision_addenda'
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
label_dir = results_root / 'pt_revision_labels' / 'label_sets'
crossfit46 = results_root / 'pt_revision_classes' / 'crossfit' / 'tables'
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            reference43 / 'conservation_index.csv', reference43 / 'class_counts_by_setting_and_contrast.csv',
            results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz', lake_file, probe_file,
            crossfit46 / 'crossfit_conservation_index.csv', crossfit46 / 'crossfit_class_counts.csv',
            *[label_dir / f'crossfit_{s}_labels_from_half{h}.csv' for s in ('i', 'ii') for h in (0, 1)],
            results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
            *[results_root / 'pt_pathway_final' / r / 'tables' / 'table_S2_pathways.csv' for r in ('scfates', 'dpt13', 'scf13_ed')],
            results_root / 'pt_revision_labels' / 'coordinate_robustness.csv']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '47.addenda.1'
SEED, N_BOOT = 43, 1000                                   # notebook 43
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.      # notebook 43
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA = .05, .5, .3, .05   # notebook 43
INDEX_MAX, INDEX_UPPER_MAX = .5, .6
TRANSPORTER_REGEX = r'^(Slc|Abc|Aqp)'
print('Results folder:', output)

## 2 · Inputs (as notebook 46, section 2)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display

import pseudospace.zonation_amplitude as amplitude_module
import pseudospace.zonation_classes as classes_module
import pseudospace.zonation_reliability as reliability_module
from pseudospace.coordinate_evaluation import sha256_folds
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.repeated_inputs import _read_and_validate_segmentation
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.zonation_amplitude import group_sums, log_ratio
from pseudospace.zonation_classes import confirm, cross_species_class, moderated_contrast, union_class, zonation_calls
from pseudospace.zonation_reliability import confirm_flat, conservation_index, paired_gradient

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__), Path(classes_module.__file__),
               Path(reliability_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
species = adata.obs.comparison_species.astype(str).to_numpy()
REVIEWED = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()
HALF = pd.Series(sha256_folds(gene_space, 2), index=gene_space)
print(f'{len(gene_space):,} genes; halves {HALF.value_counts().to_dict()}')


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']

# External references used by the primary index and the symmetric setting: mouse snRNA (male,
# female) and Lake/KPMP healthy cortex. Code as notebook 43, section 4.
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}


def census_segments(label, segment_map):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment (notebook 43's function)."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_counts, lake_totals = {}, {}
for donor in lake_donors:
    for seg_name in ('S1', 'S2', 'S3'):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
        lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
        lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)
healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']
male_sn, female_sn = sn_sex.index[sn_sex.eq('male')], sn_sex.index[sn_sex.eq('female')]


def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def donor_gradient(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
    return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])


def reference_table(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
    return moderated_contrast(per_donor, level.mean())


REF_GRADIENT = {(c, 'Lake'): donor_gradient(lake_counts, lake_totals, healthy_lake, 'S3-early' if c == 'S3c-early' else c)
                for c in CONTRASTS}
REF_GRADIENT.update({(c, 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, 'S3-early' if c == 'S3c-early' else c)
                     for c in CONTRASTS})
REF_TABLE = {(c, k): reference_table(*args, c) for c in CONTRASTS for k, args in
             (('mouse male', (sn_counts, sn_totals, male_sn)), ('mouse female', (sn_counts, sn_totals, female_sn)),
              ('human', (lake_counts, lake_totals, healthy_lake)))}
probe = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = set(probe.index[probe.probe_balance.eq('balanced')])
print(f'Lake healthy donors: {len(healthy_lake)}; mouse snRNA male/female donors: {len(male_sn)}/{len(female_sn)}')

## 3 · Notebook 46's evaluation function (unchanged) and its reproduction guards

In [ ]:
def evaluate(segment, eval_genes, tag):
    """Primary conservation index and symmetric class counts (notebook 43) for given labels and genes."""
    eval_genes = set(eval_genes)
    cortical_s3 = np.zeros(len(specimen), bool)
    for name in expected['mouse']:
        limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
        cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
    region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
    names_, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), region.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=gene_space)
    parts = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}
    pb, pb_library = {}, {}
    for name in names:
        for label, members in parts.items():
            rows = [f'{name}|{s}' for s in members if f'{name}|{s}' in frame.index]
            pb[(name, label)] = frame.loc[rows].sum()
            pb_library[(name, label)] = float(library[(specimen == name) & np.isin(region, members)].sum())
    pb = pd.DataFrame(pb)
    ours = pd.DataFrame({(contrast, name): log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
                         for contrast, (late, early) in CONTRASTS.items() for name in names}, index=gene_space)

    def count_filter(specimens, members):
        return pb[[(n, p) for n in specimens for p in members]].min(axis=1).ge(MIN_COUNTS)

    index_rows, count_rows, amplitude = [], [], {}
    for contrast, (late, early) in CONTRASTS.items():
        # Primary conservation index (notebook 43, section 5), restricted to evaluation genes.
        sets = {}
        for sp in expected:
            expressed = set(gene_space[count_filter(expected[sp], [late, early]).to_numpy()])
            sets[sp] = (paired_gradient(ours[[(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)), expressed)
        lake_g, snm_g = REF_GRADIENT[(contrast, 'Lake')], REF_GRADIENT[(contrast, 'snRNA male')]
        genes = pd.Index(sorted(sets['human'][1] & sets['mouse'][1] & lake_g[1] & snm_g[1] & eval_genes))
        index_rows.append({'labels': tag, 'contrast': contrast,
                           **conservation_index(sets['human'][0], sets['mouse'][0], lake_g[0], snm_g[0], genes, n_boot=N_BOOT, seed=SEED)})
        # Symmetric classification (notebook 43, section 6): classify all genes, count evaluation genes.
        both = gene_space[(count_filter(expected['mouse'], [late, early]) & count_filter(expected['human'], [late, early])).to_numpy()]
        tabs = {}
        for sp in expected:
            replicates = ours.loc[both, [(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)
            level = pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in expected[sp] for p in (late, early)], axis=1).mean(axis=1)
            tabs[sp] = moderated_contrast(replicates, level)
        amp_genes = [g for g in tabs['mouse'].index if g in eval_genes]
        reps = {sp: ours.loc[amp_genes, [(contrast, n) for n in expected[sp]]].to_numpy() for sp in expected}
        signal = {sp: r.mean(axis=1).var(ddof=1) - r.var(axis=1, ddof=1).mean() / r.shape[1] for sp, r in reps.items()}
        amplitude[contrast] = float(np.sqrt(max(signal['human'], 0) / signal['mouse']))
        mouse_calls, _ = zonation_calls(tabs['mouse'], alpha=ALPHA, effect=EFFECT, margin=MARGIN)
        mouse_calls = confirm(mouse_calls, REF_TABLE[(contrast, 'mouse male')], alpha=CONFIRM_ALPHA)
        human_calls, _ = zonation_calls(tabs['human'], alpha=ALPHA, effect=EFFECT * amplitude[contrast], margin=MARGIN * amplitude[contrast])
        human_calls = confirm(human_calls, REF_TABLE[(contrast, 'human')], alpha=CONFIRM_ALPHA)
        classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
        classes = confirm_flat(classes, mouse_calls, human_calls, REF_TABLE[(contrast, 'mouse male')], REF_TABLE[(contrast, 'human')],
                               alpha=CONFIRM_ALPHA, effect=EFFECT)
        counted = classes[classes.index.isin(balanced) & classes.index.isin(eval_genes)].value_counts()
        count_rows.append({'labels': tag, 'contrast': contrast, 'amplitude human/mouse': amplitude[contrast],
                           **{k: int(counted.get(k, 0)) for k in ('conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither')}})
        count_rows[-1]['mouse-only ÷ human-only'] = count_rows[-1]['mouse-only'] / max(count_rows[-1]['human-only'], 1)
    return pd.DataFrame(index_rows), pd.DataFrame(count_rows), amplitude


# Guard, every run: reviewed labels and all genes reproduce notebook 43 exactly.
check_index, check_counts, check_amplitude = evaluate(REVIEWED, gene_space, 'reviewed, all genes')
saved_index = pd.read_csv(reference43 / 'conservation_index.csv')
saved_index = saved_index[saved_index.version.str.startswith('primary')].set_index('contrast')
saved_counts = pd.read_csv(reference43 / 'class_counts_by_setting_and_contrast.csv')
saved_counts = saved_counts[saved_counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')
for row in check_index.itertuples():
    for field in ('index', 'ci_low', 'ci_high', 'genes'):
        if not np.isclose(getattr(row, field), saved_index.loc[row.contrast, field], rtol=0, atol=1e-9):
            raise ValueError(f'Index does not reproduce notebook 43: {row.contrast} {field}')
for _, row in check_counts.iterrows():
    for field in ('conserved', 'reversal', 'mouse-only', 'human-only'):
        if row[field] != saved_counts.loc[row['contrast'], field]:
            raise ValueError(f'Class counts do not reproduce notebook 43: {row["contrast"]} {field}')
print('Reproduces notebook 43 (reviewed labels, all genes): index and symmetric class counts identical.')
display(check_index.round(3))


# Guard, every run: with no exclusion, the copied function reproduces notebook 46's cross-fit setting
# (i), labels transferred on half 0 and evaluated on half 1.
labels_i0 = pd.read_csv(label_dir / 'crossfit_i_labels_from_half0.csv').set_index('structure').segment.reindex(adata.obs_names).to_numpy()
guard_index, guard_counts, _ = evaluate(labels_i0, gene_space[HALF.to_numpy() == 1], '(i) human Lake, mouse reviewed')
saved46 = pd.read_csv(crossfit46 / 'crossfit_conservation_index.csv')
saved46 = saved46[saved46.labels.str.startswith('(i)') & saved46['labels from half'].eq(0)].set_index('contrast')
for _, row in guard_index.iterrows():
    for field in ('index', 'ci_low', 'ci_high', 'genes'):
        if not np.isclose(row[field], saved46.loc[row['contrast'], field], rtol=0, atol=1e-9):
            raise ValueError(f'Does not reproduce notebook 46: {row["contrast"]} {field}')
print('Reproduces notebook 46 (setting (i), labels from half 0): identical index and interval.')

## 4 · (a) Without the 611 integration genes

The 611 genes are notebook 13's pass-2 PCA genes: Harmony HVGs that are also pass 1's top-2000
HVGs. They built the embedding whose clusters became the reviewed labels. Every evaluation below
drops them from the evaluation genes; classification still uses all genes for the moderation
prior, exactly as in notebooks 43 and 46.

In [ ]:
pass2 = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
         if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(pass2) != 1:
    raise FileNotFoundError('Expected one notebook 13 pass-2 cache.')
pass2_var = ad.read_h5ad(str(pass2[0])[:-len('.key.json')], backed='r').var
INTEGRATION = set(pass2_var.index[(pass2_var.highly_variable_for_harmony & pass2_var.highly_variable).to_numpy(bool)])
if len(INTEGRATION) != 611:
    raise ValueError(f'Expected 611 integration genes, found {len(INTEGRATION)}')
print(f'{len(INTEGRATION & set(gene_space))} of the 611 integration genes are in the gene space.')
evaluations = {('reviewed', 'all genes'): (REVIEWED, gene_space)}
for setting, stem in (('(i) human Lake, mouse reviewed', 'i'), ('(ii) human Lake, mouse microdissection', 'ii')):
    for half in (0, 1):
        seg = pd.read_csv(label_dir / f'crossfit_{stem}_labels_from_half{half}.csv').set_index('structure').segment.reindex(adata.obs_names).to_numpy()
        evaluations[(setting, f'labels from half {half}, evaluated on half {1 - half}')] = (seg, gene_space[HALF.to_numpy() == 1 - half])
index_parts, count_parts = [], []
for (setting, genes_label), (seg, genes) in evaluations.items():
    kept = pd.Index([g for g in genes if g not in INTEGRATION])
    idx, cnt, _ = evaluate(seg, kept, setting)
    index_parts.append(idx.assign(evaluation=genes_label, **{'integration genes removed': len(genes) - len(kept)}))
    count_parts.append(cnt.assign(evaluation=genes_label))
INDEX = pd.concat(index_parts, ignore_index=True)
COUNTS = pd.concat(count_parts, ignore_index=True)
INDEX.to_csv(tables / 'no_integration_genes_conservation_index.csv', index=False)
COUNTS.to_csv(tables / 'no_integration_genes_class_counts.csv', index=False)
display(INDEX[['labels', 'evaluation', 'contrast', 'index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human',
               'ceiling_mouse', 'genes', 'integration genes removed']].round(3))
display(COUNTS[['labels', 'evaluation', 'contrast', 'amplitude human/mouse', 'conserved', 'reversal', 'mouse-only',
                'human-only', 'mouse-only ÷ human-only']].round(2))
primary = INDEX[INDEX.contrast.isin(['S2-S1', 'S3-early'])]
passes = (primary['index'] <= INDEX_MAX) & (primary.ci_high < INDEX_UPPER_MAX)
CLAIM_STANDS = bool(passes.all())
print(f'(a) Claim "largely species-specific" without integration genes: {"stands" if CLAIM_STANDS else "weakened"} '
      f'(max index {primary["index"].max():.3f}, max upper bound {primary.ci_high.max():.3f})')

## 5 · (b) Transporter shares among zonated genes

Classes are notebook 43's reviewed symmetric classes, union of S2 − S1 and S3c − early, on
probe-balanced genes. A gene is "zonated" if its class is conserved, reversal, mouse-only or
human-only; the share is computed over zonated genes. The family is `^(Slc|Abc|Aqp)` on mouse
symbols, case-sensitive.

In [ ]:
symmetric = pd.read_csv(results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv', index_col=0)
union_column = 'union S2-S1 + S3c-early'
balanced_classes = symmetric[symmetric.probe_balanced.astype(bool)]
ZONATED = ['conserved', 'reversal', 'mouse-only', 'human-only']
family = balanced_classes.index.str.match(TRANSPORTER_REGEX)
rows = []
for label, frame in (('Slc/Abc/Aqp', balanced_classes[family]), ('all genes', balanced_classes)):
    counts_ = frame[union_column].value_counts()
    zonated = int(sum(counts_.get(k, 0) for k in ZONATED))
    rows.append({'genes': label, 'probe-balanced genes': len(frame), **{k: int(counts_.get(k, 0)) for k in ZONATED},
                 'zonated': zonated, 'conserved share of zonated': counts_.get('conserved', 0) / zonated if zonated else np.nan})
TRANSPORTERS = pd.DataFrame(rows)
TRANSPORTERS.to_csv(tables / 'transporter_class_shares.csv', index=False)
balanced_classes[family & balanced_classes[union_column].isin(ZONATED)][[union_column]].to_csv(tables / 'transporter_zonated_genes.csv')
display(TRANSPORTERS.round(3))

## 6 · (c) Table S2 with coordinate robustness

Notebook 37's scfates Table S2 (every tested pathway) is joined one-to-one on `pathway_id` with
robust, core and peak position from the DPT13 and SCF13-ED runs of notebook 37. Sign agreement and
the notebook-43 category come from `coordinate_robustness.csv`, which covers the union of the
scfates and DPT13 robust lists.

In [ ]:
base = pd.read_csv(results_root / 'pt_pathway_final' / 'scfates' / 'tables' / 'table_S2_pathways.csv')
merged = base.copy()
for label, run in (('DPT13', 'dpt13'), ('SCF13-ED', 'scf13_ed')):
    other = pd.read_csv(results_root / 'pt_pathway_final' / run / 'tables' / 'table_S2_pathways.csv')
    if other.pathway_id.duplicated().any():
        raise ValueError(f'{label}: duplicate pathway ids.')
    other = other[['pathway_id', 'robust', 'core', 'peak_position', 'human_high_at_peak', 'z_joint']].rename(
        columns={c: f'{c} {label}' for c in ('robust', 'core', 'peak_position', 'human_high_at_peak', 'z_joint')})
    merged = merged.merge(other, on='pathway_id', how='left', validate='one_to_one')
    for column in (f'robust {label}', f'core {label}'):
        merged[column] = merged[column].fillna(False).astype(bool)
robustness = pd.read_csv(results_root / 'pt_revision_labels' / 'coordinate_robustness.csv')
merged = merged.merge(robustness[['pathway_id', 'direction agreement (where defined)', 'notebook 43 category']],
                      on='pathway_id', how='left', validate='one_to_one')
if len(merged) != len(base) or merged.pathway_id.duplicated().any():
    raise ValueError('Table S2 join is not one-to-one.')
# Guard: coordinate_robustness.csv agrees with the run tables on the union.
check = robustness.set_index('pathway_id').join(merged.set_index('pathway_id'), how='left', rsuffix='_s2')
for column in ('robust DPT13', 'core DPT13', 'robust SCF13-ED', 'core SCF13-ED'):
    if not (check[column].astype(bool) == check[column + '_s2'].astype(bool)).all():
        raise ValueError(f'coordinate_robustness.csv disagrees with notebook 37 on {column}')
for label in ('DPT13', 'SCF13-ED'):
    a, b = check[f'peak {label}'], check[f'peak_position {label}']
    if not np.allclose(a.dropna(), b.loc[a.dropna().index], atol=1e-12):
        raise ValueError(f'coordinate_robustness.csv disagrees with notebook 37 on peak {label}')
merged.to_csv(output / 'table_S2_pathways_with_coordinate.csv', index=False)
print(f'Table S2 with coordinate columns: {len(merged):,} pathways; robust under scfates/DPT13/SCF13-ED: '
      f'{int(merged.robust.sum())}/{int(merged["robust DPT13"].sum())}/{int(merged["robust SCF13-ED"].sum())}')

## 7 · (b′) Transporters: direction agreement among genes zonated in both species

The referee's point: transporters are strongly expressed and strongly zonated, so they are more
often called zonated in both species (power). The comparison that separates conservation from
power is direction agreement among genes zonated in both species (conserved ÷ (conserved +
reversal)).

In [ ]:
def agreement(frame, label):
    c = frame[union_column].value_counts()
    zonated = int(sum(c.get(k, 0) for k in ZONATED))
    both = int(c.get('conserved', 0) + c.get('reversal', 0))
    return {'genes': label, 'zonated': zonated, 'zonated in both species': both,
            'zonated-in-both rate': both / zonated if zonated else np.nan,
            'conserved': int(c.get('conserved', 0)), 'reversal': int(c.get('reversal', 0)),
            'direction agreement among zonated in both': c.get('conserved', 0) / both if both else np.nan}


AGREEMENT = pd.DataFrame([agreement(balanced_classes[family], 'Slc/Abc/Aqp'), agreement(balanced_classes, 'all genes')])
conserved_all = balanced_classes[union_column].eq('conserved')
AGREEMENT['transporter share of the conserved class'] = [(family & conserved_all).sum() / conserved_all.sum(), np.nan]
AGREEMENT.to_csv(tables / 'transporter_direction_agreement.csv', index=False)
display(AGREEMENT.round(3))

## 8 · (d) Integration-gene enrichment under the symmetric classes

The earlier "16% against 2%" used notebook 40's classes. Here the 611 PCA genes and the 752
Harmony HVGs are counted per notebook-43 symmetric union class.

In [ ]:
HVG752 = set(pass2_var.index[pass2_var.highly_variable_for_harmony.to_numpy(bool)])
enrichment_rows = []
for scope, frame in (('all classified genes', symmetric), ('probe-balanced', balanced_classes)):
    for set_label, genes_ in (('611 PCA genes', INTEGRATION), ('752 Harmony HVGs', HVG752)):
        member = frame.index.isin(sorted(genes_))
        for klass, size in frame[union_column].value_counts().items():
            in_class = frame[union_column].eq(klass)
            enrichment_rows.append({'scope': scope, 'gene set': set_label, 'class': klass, 'class size': int(size),
                                    'integration genes': int((in_class & member).sum()),
                                    'share of class': float((in_class & member).sum() / size)})
ENRICHMENT = pd.DataFrame(enrichment_rows)
ENRICHMENT.to_csv(tables / 'integration_genes_by_symmetric_class.csv', index=False)
display(ENRICHMENT.pivot_table(index='class', columns=['scope', 'gene set'], values='share of class').round(3))
zonated_integration = symmetric[symmetric.index.isin(sorted(INTEGRATION)) & symmetric[union_column].isin(ZONATED)][union_column]
print('Zonated integration genes by class:', zonated_integration.value_counts().to_dict())

## 9 · (e) What the robust list is made of, and where its peaks fall in segment units

Each robust pathway is checked against two joint tests from the same notebook-37 run: the
smooth position beyond S1/S2/S3 steps (`T_position_given_segments`) and the step screen
(`T_discrete_spatial`); a call is q_joint ≤ 0.05 in the species partition. Peaks are converted to
segment units with the run's mean segment transitions: S1 is [0, t12), S2 [t12, t23), S3 [t23, 1];
the value is the segment index plus the fraction through it (1.4 = 40% through S2).

In [ ]:
RUNS = {'scFates': 'scfates', 'DPT13': 'dpt13', 'SCF13-ED': 'scf13_ed'}


def segment_units(position, t12, t23):
    position = np.asarray(position, float)
    return np.where(position < t12, position / t12,
                    np.where(position < t23, 1 + (position - t12) / (t23 - t12), 2 + (position - t23) / (1 - t23)))


run_tables, decomposition_rows, peak_frames = {}, [], {}
for label, run in RUNS.items():
    folder = results_root / 'pt_pathway_final' / run / 'tables'
    table = pd.read_csv(folder / 'table_S2_pathways.csv').set_index('pathway_id')
    tests = pd.read_csv(folder / 'matched_and_joint_tests_all_partitions.csv')
    tests = tests[tests.partition.eq('species')]
    called = {stat: set(tests.loc[tests.statistic.eq(stat) & tests.q_joint.le(.05), 'pathway_id'])
              for stat in ('T_position_given_segments', 'T_discrete_spatial', 'T_spatial')}
    transitions = pd.read_csv(folder / 'segment_transitions_by_specimen.csv')
    pooled = transitions.groupby('transition').position.mean()
    by_species = transitions.groupby(['species', 'transition']).position.mean()
    units = pd.DataFrame({'peak': table.peak_position,
                          'segment units (pooled transitions)': segment_units(table.peak_position, pooled['S1→S2'], pooled['S2→S3'])},
                         index=table.index)
    for sp in ('human', 'mouse'):
        units[f'segment units ({sp} transitions)'] = segment_units(table.peak_position, by_species[(sp, 'S1→S2')], by_species[(sp, 'S2→S3')])
    units['human-high at peak'] = table.human_high_at_peak
    peak_frames[label] = units
    run_tables[label] = table
    for level in ('robust', 'core'):
        members = set(table.index[table[level].astype(bool)])
        decomposition_rows.append({'coordinate': label, 'set': level, 'pathways': len(members),
                                   'called beyond S1/S2/S3 steps': len(members & called['T_position_given_segments']),
                                   'called by S1/S2/S3 step screen': len(members & called['T_discrete_spatial']),
                                   'called by both': len(members & called['T_position_given_segments'] & called['T_discrete_spatial']),
                                   'mean S1→S2 / S2→S3 transition': f"{pooled['S1→S2']:.3f} / {pooled['S2→S3']:.3f}"})
DECOMPOSITION = pd.DataFrame(decomposition_rows)
DECOMPOSITION.to_csv(tables / 'robust_list_decomposition.csv', index=False)
display(DECOMPOSITION)

robust36 = run_tables['scFates'].index[run_tables['scFates'].robust.astype(bool)]
peaks = pd.concat({label: frame.reindex(robust36) for label, frame in peak_frames.items()}, axis=1)
peaks.insert(0, 'name', run_tables['scFates'].loc[robust36, 'name'])
peaks.insert(1, 'core', run_tables['scFates'].loc[robust36, 'core'].astype(bool))
for label in ('DPT13', 'SCF13-ED'):
    peaks[(label, 'robust here')] = run_tables[label].robust.reindex(robust36).fillna(False).astype(bool).to_numpy()
peaks.to_csv(tables / 'robust_peaks_in_segment_units.csv')
peak_rows = []
for label in ('DPT13', 'SCF13-ED'):
    for subset, mask in (('scFates robust with a peak in the other run', peaks[(label, 'peak')].notna()),
                         ('robust under both', peaks[(label, 'robust here')] & peaks[(label, 'peak')].notna())):
        a, b = peaks.loc[mask, ('scFates', 'peak')], peaks.loc[mask, (label, 'peak')]
        ua, ub = peaks.loc[mask, ('scFates', 'segment units (pooled transitions)')], peaks.loc[mask, (label, 'segment units (pooled transitions)')]
        peak_rows.append({'comparison': f'scFates vs {label}', 'pathways': subset, 'n': int(mask.sum()),
                          'Spearman, coordinate': a.corr(b, method='spearman'),
                          'Spearman, segment units': ua.corr(ub, method='spearman'),
                          'moved > 0.25 coordinate units': int(((a - b).abs() > .25).sum()),
                          'segment changed': int((np.floor(ua) != np.floor(ub)).sum())})
PEAK_AGREEMENT = pd.DataFrame(peak_rows)
PEAK_AGREEMENT.to_csv(tables / 'peak_agreement_between_coordinates.csv', index=False)
display(PEAK_AGREEMENT.round(3))
for label in RUNS:
    block = peak_frames[label].reindex(robust36)
    for high, name in ((True, 'human-high'), (False, 'mouse-high')):
        part = block[block['human-high at peak'].astype(bool) == high]['segment units (pooled transitions)']
        print(f'{label}: {name} median peak in segment units {part.median():.2f} (n={part.notna().sum()})')

## 10 · (f) Reversal genes under each label setting

Reversals (zonated in both species in opposite directions) in S2 − S1 and S3c − early, symmetric
setting, probe-balanced genes, from notebooks 43, 46 and this notebook.

In [ ]:
reversal_rows = []
for tag, description in (('reviewed', 'reviewed labels, all genes'),
                         ('human_lake_centred__mouse_reviewed', 'transferred (full-gene, circular): human Lake, mouse reviewed'),
                         ('human_lake_centred__mouse_micro_centred', 'transferred (full-gene, circular): human Lake, mouse microdissection')):
    frame = pd.read_csv(results_root / 'pt_revision_classes' / tag / 'tables' / 'class_counts_by_setting_and_contrast.csv')
    frame = frame[frame.setting.eq('symmetric (A3b + A8)')].set_index('contrast')
    reversal_rows.append({'labels': description, 'evaluation genes': 'all',
                          **{f'{c} reversal': int(frame.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                          **{f'{c} conserved': int(frame.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
cross = pd.read_csv(crossfit46 / 'crossfit_class_counts.csv')
for (setting, half), block in cross.groupby(['labels', 'labels from half'], sort=False):
    block = block.set_index('contrast')
    reversal_rows.append({'labels': f'cross-fit {setting}', 'evaluation genes': f'half {1 - half} (labels from half {half})',
                          **{f'{c} reversal': int(block.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                          **{f'{c} conserved': int(block.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
for (setting, evaluation), block in COUNTS.groupby(['labels', 'evaluation'], sort=False):
    block = block.set_index('contrast')
    reversal_rows.append({'labels': f'{setting}, without the 611 integration genes', 'evaluation genes': evaluation,
                          **{f'{c} reversal': int(block.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                          **{f'{c} conserved': int(block.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
REVERSALS = pd.DataFrame(reversal_rows)
REVERSALS.to_csv(tables / 'reversal_label_sensitivity.csv', index=False)
display(REVERSALS)

summary = {'logic_version': NOTEBOOK_LOGIC_VERSION, 'claim stands without integration genes': CLAIM_STANDS,
           'max index': float(primary['index'].max()), 'max upper bound': float(primary.ci_high.max()),
           'transporters': json.loads(TRANSPORTERS.to_json(orient='records')),
           'table_S2_rows': len(merged),
           'agreement': json.loads(AGREEMENT.to_json(orient='records')),
           'decomposition': json.loads(DECOMPOSITION.to_json(orient='records')),
           'peak_agreement': json.loads(PEAK_AGREEMENT.to_json(orient='records'))}
(output / 'run_manifest.json').write_text(json.dumps(summary, indent=2, default=str))